# 5 — Neural Network Models (Updated with Fixes & Tuning)

**Question.** Does the wording of a company's Item 7 (management's discussion) predict its share-price volatility over the next 12 months, beyond what last year's volatility already tells us?

### Summary of Fixes & Tuning Changes in This Version
Every modification below addresses a specific methodological, mathematical, or engineering issue identified from the initial `05-MLP.ipynb` run:

1. **Full-Window Refit for the MLP (Eliminating the 1–2 Year Data Handicap):** In the initial version, Ridge tuned its penalty on `validate_year` and then refitted on the full training window (`tr_idx`), whereas the MLP stopped on `validate_year` and was never refitted. In the `strict` window (which already leaves a 1-year gap before `test_year`), holding out `validate_year` left the MLP with a 2-year regime gap. The MLP now uses `validate_year` to choose the optimal epoch count and then refits on the full training window (`tr_idx`) for that exact number of epochs.
2. **Expanded Ridge Regularization Grid:** In the initial run, `ridge: words + last year` hit the upper ceiling `alpha = 10` on 11 of 12 splits. `ALPHAS` is expanded to `[0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]` so the linear benchmark is properly tuned.
3. **Leakage-Free Validation Feature Building:** Previously `build_features` fit the 5,000-word vocabulary and IDF weights on all of `train` (including `validate_year`) before tuning. Features are now built strictly on `fit_idx` during tuning/early stopping, and rebuilt on `tr_idx` for the final full-window refit.
4. **Targeted Weight Decay (Optimizer Parameter Groups):** `AdamW` previously applied `weight_decay` to every parameter, including `self.skip.weight` and biases, pulling the persistence slope away from `1.0`. Parameters are now split into two groups so `weight_decay=0.0` is applied to the skip connection and all bias terms.
5. **Capacity & Regularization Tuning (Addressing Over-Parameterization):** With 5,000 input features and `HIDDEN = 64`, the original network had ~320k parameters against ~7.5k–14k training reports. Section 5.5 adds a validation-year hyperparameter sweep over hidden width (`16, 32, 64`), dropout (`0.3, 0.5`), and weight decay (`1e-2, 1e-1`).
6. **Within-Year Demeaned MLP & Robust Huber Loss Variants:** The initial level/spread decomposition and `Gradient × Input` attributions showed that pooled models mostly gained by guessing the macro year from accounting-standard words (`stockbased`, `eitf`, `fin`), while 0.3% takeover outliers (`logvol_after < -5`) drove >11% of total MSE. We add two targeted MLP variants:
   - **Huber-loss MLP** (`nn.HuberLoss`) to damp outlier gradients during training while still scoring on standard MSE.
   - **Within-year demeaned MLP** (predicting cross-sectional residuals relative to each year's mean) to test genuine company-level spread prediction.
7. **Consistent Takeover Evaluation & Ensemble Attribution:** Restored the missing `copy_vs_line` / `alex` verification cell so the notebook runs cleanly top-to-bottom; fixed the takeover table to score the 3-seed averaged ensembles (matching the main summary table); and updated `Gradient × Input` attribution to clear gradients cleanly and average across all 3 seeds.
8. **Expanded Scaling Benchmark:** Added a CPU thread-scaling benchmark (`torch.set_num_threads`) alongside the data-volume learning curve to directly measure parallel vs. single-thread performance for the 1,000× scaling analysis.

*(Conclusions and interpretations will be written in Section 5.8 after running the updated notebook.)*

## 5.1 — Data and Requirements

In [ ]:
import gzip
import json
import pickle
import sys
import tarfile
import time
import urllib.request
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import Ridge
import torch
from torch import nn

ROOT = Path("..")                              # this notebook lives in a personal folder, so the repo root is one level up
PROCESSED = ROOT / "data" / "processed"
OUTPUT = ROOT / "data" / "output"
kogan_dir = ROOT / "data" / "raw" / "kogan"
edgar_dir = PROCESSED / "edgar_matched"
KOGAN_URL = "https://www.cs.cmu.edu/~ark/10K/data/"
BOW_DIR = PROCESSED / "bow"                    # word counts for every report, rebuilt if deleted
BOW_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "Alex"))         # scoring.py: the same file 02-Linear-models imports
import scoring

# --- split: supplied by scoring.masks(df, test_year, window), identical to 02 --------------------
MAIN_WINDOW = "paper"                          # single-window views (tuning, words, cost, scatter)

# --- features ------------------------------------------------------------------------------------
N_WORDS = 5000                                 # vocabulary size
MIN_DF = 50                                    # a word must appear in at least 50 training reports
TOKEN = r"(?u)\b[a-z]{2,}\b"                   # lower-case words of 2+ letters

# --- ridge penalty grid (FIX: expanded upward from [0.1..10], which hit the ceiling on 11/12 splits)
ALPHAS = [0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]

# --- default network hyperparameters (overridden if tuning in Section 5.5 selects better values) --
HIDDEN = 32                                    # reduced from 64 to curb overfitting (~160k vs ~320k params)
DROPOUT = 0.5                                  # increased from 0.3 for high-dimensional sparse text
LR = 1e-3
WEIGHT_DECAY = 5e-2                            # stronger L2 on text weights only (skip & biases excluded)
BATCH = 256
MAX_EPOCHS = 40
PATIENCE = 5                                   # stop if validation error has not improved for 5 epochs
HUBER_DELTA = 1.0                              # Huber transition threshold in standardized z-units
SEEDS = [0, 1, 2]
NET_VARIANTS = {False: "words only", True: "words + last year"}

VERY_LOW = -5.0                                # takeover-type reports, as in 04-Volatility-Checking
device = "cpu"                                 # GPU not needed at this size; see Section 5.9 for scaling

## 5.2 — Reports and the Split

Every report has `logvol_before` (12 months before filing) and `logvol_after` (12 months after), both from Kogan. We predict `logvol_after`. The reports, their order, and their labels come from `data.pkl` through `scoring.frame`; the Item 7 texts are matched by report key.

The split is strictly by **filing year**, not random, so market-wide volatility cannot leak across reports from the same year. For each `(window, test_year)`, `scoring.masks` gives the training and test masks, and the last training year (`validate_year`) is held out for hyperparameter tuning and early stopping.

In [ ]:
with open(PROCESSED / "data.pkl", "rb") as f:
    d = pickle.load(f)
df = scoring.frame(d)
y = df["logvol_after"].to_numpy()
prev = df["logvol_before"].to_numpy()
year_arr = df["year"].to_numpy()
print(f"{len(df):,} reports, {df.year.min()}-{df.year.max()}")

def as_mask(m):
    m = np.asarray(m)
    if m.dtype == bool:
        return m
    out = np.zeros(len(df), dtype=bool)
    out[m] = True
    return out

# --- Item 7 source for every report in df ---------------------------------------------------------
meta = pd.read_csv(PROCESSED / "dataset.csv", parse_dates=["date"]).drop(columns="item7_source", errors="ignore")
assert meta["key"].is_unique, "key must identify one report"
src_file = PROCESSED / "item7_source.csv"
USE_EDGAR = src_file.exists() and edgar_dir.exists() and any(edgar_dir.glob("*.jsonl.gz"))
TEXT_TAG = "edgar" if USE_EDGAR else "kogan"
if USE_EDGAR:
    meta = meta.merge(pd.read_csv(src_file)[["key", "item7_source"]], on="key", how="inner")
    assert not meta.duplicated(["cik", "year"]).any(), "company + year must identify one report"
else:
    meta["item7_source"] = "kogan"

data = df[["key"]].merge(meta, on="key", how="left", validate="one_to_one")
missing = data["item7_source"].isna()
assert not missing.any(), f"{int(missing.sum())} reports in data.pkl have no Item 7 source"
assert (data["year"].to_numpy() == year_arr).all(), "filing years differ between data.pkl and dataset.csv"
assert np.allclose(data["logvol_after"], y) and np.allclose(data["logvol_before"], prev), "labels differ"
print("Item 7 source:", data["item7_source"].value_counts().to_dict(),
      "(EDGAR rule from 03)" if USE_EDGAR else "(Kogan only: item7_source.csv / edgar_matched not found)")

In [ ]:
rows = []
for window in scoring.WINDOWS:
    for T in scoring.TEST_YEARS:
        train, test = (as_mask(m) for m in scoring.masks(df, T, window))
        ty = sorted(set(year_arr[train]))
        rows.append({"window": window, "test_year": T, "train_years": f"{ty[0]}-{ty[-1]}",
                     "validate_year": ty[-1], "train_reports": int(train.sum()), "test_reports": int(test.sum())})
split_table = pd.DataFrame(rows)

shared = pd.read_csv(OUTPUT / "splits.csv")
for r in shared.itertuples():
    train, _ = scoring.masks(df, r.test_year, r.window)
    assert as_mask(train).sum() == r.train_reports
assert set(shared.window) == set(scoring.WINDOWS)
print("The windows match data/output/splits.csv")
split_table

**Reproducibility Check (Restored):** The cell below scores `copy last year` and the straight-line baseline with `scoring.evaluate` and checks exact alignment with `alex-02-ridge-results.csv`. Restoring this cell ensures `copy_vs_line` and `alex` are always defined when running top-to-bottom from a fresh kernel.

In [ ]:
copy_vs_line = scoring.evaluate(df, scoring.predict_copy, name="copy last year")

alex_file = OUTPUT / "alex-02-ridge-results.csv"
if alex_file.exists():
    alex = pd.read_csv(alex_file)
    keys_ = ["window", "test_year"]
    theirs = alex[alex.model == "copy last year"].set_index(keys_)[["test_reports", "mse", "baseline_mse"]].sort_index()
    ours = copy_vs_line.set_index(keys_)[["test_reports", "mse", "baseline_mse"]].sort_index()
    pd.testing.assert_frame_equal(ours, theirs, check_dtype=False)
    print("Same test reports and same copy-last-year and straight-line MSEs as 02 in every window and test year.")
else:
    alex = None
    print(f"{alex_file.name} not found: only splits.csv checks alignment with 02.")

copy_vs_line[copy_vs_line.window == MAIN_WINDOW][["test_year", "test_reports", "mse", "baseline_mse"]].round(4).rename(
    columns={"mse": "copy last year", "baseline_mse": "straight line"}).set_index("test_year")

In [ ]:
def annual(logvol):
    return np.exp(logvol) * np.sqrt(252)

by_year = df.groupby("year").agg(
    reports=("key", "size"),
    typical_vol_before=("logvol_before", lambda v: annual(v.mean())),
    typical_vol_after=("logvol_after", lambda v: annual(v.mean())),
    corr_before_after=("logvol_before", lambda v: np.corrcoef(v, df.loc[v.index, "logvol_after"])[0, 1]),
)
by_year.style.format({"typical_vol_before": "{:.0%}", "typical_vol_after": "{:.0%}", "corr_before_after": "{:.2f}"})

## 5.3 — Text to Word Counts

Candidate word counts (`min_df = 50` across all reports, English stop words removed) are computed once and cached to `data/processed/bow/`. Because any word appearing in at least 50 training reports must also appear in at least 50 total reports, this global candidate menu introduces zero leakage. Within each split, the 5,000-word vocabulary and sublinear TF-IDF weights are then fitted strictly on the fitting reports only:

$$x_{ij} \propto \big(1 + \ln c_{ij}\big)\cdot \mathrm{idf}_j,\qquad \mathrm{idf}_j = \ln\frac{1+n}{1+\mathrm{df}_j} + 1,$$

with each row L2-normalized to unit length.

In [ ]:
def kogan_file(year):
    f = kogan_dir / f"{year}.tok.tgz"
    if not f.exists():
        kogan_dir.mkdir(parents=True, exist_ok=True)
        print("downloading", f.name)
        urllib.request.urlretrieve(KOGAN_URL + f.name, f)
    return f

def load_year_texts(year):
    rows = data[data["year"] == year]
    want_edgar = {(int(r.cik), int(r.year)): r.key for r in rows.itertuples() if r.item7_source == "edgar"}
    want_kogan = {r.key for r in rows.itertuples() if r.item7_source == "kogan"}
    texts = {}
    if want_edgar:
        with gzip.open(edgar_dir / f"{year}.jsonl.gz", "rt") as f:
            for line in f:
                rec = json.loads(line)
                key = want_edgar.get((int(rec["cik"]), int(rec["year"])))
                if key is not None:
                    texts[key] = rec["section_7"]
    if want_kogan:
        with tarfile.open(kogan_file(year)) as tar:
            for m in tar:
                if m.isfile():
                    key = Path(m.name).name.split(".")[0]
                    if key in want_kogan:
                        texts[key] = tar.extractfile(m).read().decode("utf-8", "replace")
    missing = set(rows["key"]) - set(texts)
    assert not missing, f"{year}: no text found for {len(missing)} reports, e.g. {sorted(missing)[:2]}"
    return [texts[k] for k in rows["key"]]

keys = df["key"].tolist()
years_sorted = sorted(int(v) for v in np.unique(year_arr))
order = np.concatenate([np.flatnonzero(year_arr == yr) for yr in years_sorted])

cache_npz, cache_json = BOW_DIR / f"candidates_{TEXT_TAG}.npz", BOW_DIR / f"candidates_{TEXT_TAG}.json"
cached = None
if cache_npz.exists() and cache_json.exists():
    cached = json.loads(cache_json.read_text())
    if cached["keys"] != keys:
        cached = None
        print("saved word counts are for a different set of reports: counting again")
if cached is not None:
    X_counts = sp.load_npz(cache_npz).tocsr()
    vocab, timings = cached["vocab"], cached["timings"]
    print("loaded saved word counts")
else:
    def all_texts():
        for yr in years_sorted:
            yield from load_year_texts(yr)
    t0 = time.perf_counter()
    vec = CountVectorizer(min_df=MIN_DF, stop_words="english", token_pattern=TOKEN, dtype=np.int32)
    X_sorted = vec.fit_transform(all_texts()).tocsr()
    X_counts = X_sorted[np.argsort(order)]
    timings = {"count_seconds": time.perf_counter() - t0, "n_reports": int(X_counts.shape[0])}
    vocab = [str(w) for w in vec.get_feature_names_out()]
    sp.save_npz(cache_npz, X_counts)
    cache_json.write_text(json.dumps({"keys": keys, "vocab": vocab, "timings": timings}))

assert X_counts.shape == (len(df), len(vocab)), "row order must match `df`"
vocab = np.array(vocab)
print("count matrix (reports x candidate words):", X_counts.shape, "| timings:", {k: round(v, 1) for k, v in timings.items()})

## 5.4 — Leakage-Free Features, Split Structure & Expanded Ridge Baseline

### Reasons for Changes in This Section
- **Strict validation isolation (`X_tune` vs. `X_full`):** Previously `build_features(train, test)` computed vocabulary selection and IDF weights using the entire `train` window (including `validate_year`). Here `build_features(fit_mask, transform_mask)` builds two clean feature matrices:
  1. `X_tune` fitted **only on `fit_idx`** (training years excluding `validate_year`) for penalty selection and early stopping.
  2. `X_full` fitted **on `tr_idx`** (all training years in the window) for the final refitted Ridge and MLP models evaluated on `test_year`.
- **Expanded `ALPHAS`:** Extending Ridge penalties up to `1000` resolves the ceiling effect where `alpha = 10` was selected on 11/12 splits in the initial run.
- **Within-year demeaning helper:** For the within-year MLP variant, we compute each training year's mean residual $r_{i} = (y_i - \hat{y}_{i,\text{line}}) - \bar{r}_{\text{year}(i)}$, preventing the text branch from exploiting macro-level shifts across years.

In [ ]:
def mse(a, b):
    return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

def split_key(train, test):
    return (tuple(sorted(int(v) for v in set(year_arr[as_mask(train)]))), int(year_arr[as_mask(test)][0]))

def build_features(fit_mask, transform_mask):
    """Fit vocabulary and sublinear TF-IDF strictly on fit_mask; transform transform_mask."""
    fit_idx = np.flatnonzero(fit_mask)
    counts_fit = X_counts[fit_idx]
    in_reports = np.asarray((counts_fit > 0).sum(axis=0)).ravel()
    total = np.asarray(counts_fit.sum(axis=0)).ravel()
    eligible = np.flatnonzero(in_reports >= MIN_DF)
    cols = np.sort(eligible[np.argsort(-total[eligible], kind="stable")[:N_WORDS]])

    rows = np.flatnonzero(fit_mask | transform_mask)
    tfidf = TfidfTransformer(sublinear_tf=True).fit(counts_fit[:, cols])
    X = tfidf.transform(X_counts[rows][:, cols]).astype(np.float32).tocsr()
    pos = np.full(len(df), -1, dtype=np.int64)
    pos[rows] = np.arange(len(rows))
    return X, pos, cols

def demean_by_year(values, idx):
    """Subtract each year's mean within idx so the target captures within-year spread only."""
    out = np.zeros(len(df), dtype=np.float32)
    for yr in np.unique(year_arr[idx]):
        m = idx[year_arr[idx] == yr]
        out[m] = (values[m] - values[m].mean()).astype(np.float32)
    return out

def make_split(train, test):
    train, test = as_mask(train), as_mask(test)
    assert len(set(year_arr[test])) == 1, "one test year at a time"
    years = sorted(int(v) for v in set(year_arr[train]))
    val_year = years[-1]

    fit_mask = train & (year_arr != val_year)
    val_mask = train & (year_arr == val_year)
    tr_idx, te_idx = np.flatnonzero(train), np.flatnonzero(test)
    fit_idx, val_idx = np.flatnonzero(fit_mask), np.flatnonzero(val_mask)

    # Leakage-free features: tune strictly on fit_mask; final refit on full train mask
    X_tune, pos_tune, cols_tune = build_features(fit_mask, val_mask)
    X_full, pos_full, cols_full = build_features(train, test)

    # Standardization statistics for tuning (fit_idx) and for final refit (tr_idx)
    mu_fit, sd_fit = float(y[fit_idx].mean()), float(y[fit_idx].std())
    mu_full, sd_full = float(y[tr_idx].mean()), float(y[tr_idx].std())

    # Straight-line baseline predictions (for tuning and final refit)
    a_all, b_all = scoring.fit_line(df, train)
    a_fit, b_fit = scoring.fit_line(df, fit_mask)
    line_all = a_all + b_all * prev
    line_fit = a_fit + b_fit * prev

    # Within-year demeaned residuals on top of the straight line (removes macro year level)
    resid_fit = y - line_fit
    resid_all = y - line_all
    wy_resid_fit = demean_by_year(resid_fit, fit_idx)
    wy_resid_val = demean_by_year(resid_fit, val_idx)
    wy_resid_all = demean_by_year(resid_all, tr_idx)
    # Combined array for tuning (fit_idx demeaned by fit years, val_idx demeaned by val_year)
    wy_resid_tune = wy_resid_fit.copy()
    wy_resid_tune[val_idx] = wy_resid_val[val_idx]

    return SimpleNamespace(
        train=train, test=test, fit_mask=fit_mask, val_mask=val_mask,
        years=tuple(years), val_year=val_year, test_year=int(year_arr[test][0]),
        tr_idx=tr_idx, te_idx=te_idx, fit_idx=fit_idx, val_idx=val_idx,
        X_tune=X_tune, pos_tune=pos_tune, cols_tune=cols_tune,
        X_full=X_full, pos_full=pos_full, cols_full=cols_full,
        mu_fit=mu_fit, sd_fit=sd_fit,
        mu_full=mu_full, sd_full=sd_full,
        y_z_fit=((y - mu_fit) / sd_fit).astype(np.float32),
        prev_z_fit=((prev - mu_fit) / sd_fit).astype(np.float32),
        y_z_full=((y - mu_full) / sd_full).astype(np.float32),
        prev_z_full=((prev - mu_full) / sd_full).astype(np.float32),
        line_fit=line_fit, line_all=line_all,
        wy_resid_tune=wy_resid_tune, wy_resid_all=wy_resid_all,
    )

def fit_ridge(S, tune_target, final_target):
    """Tune alpha on fit_idx -> val_idx using X_tune; refit on full tr_idx using X_full."""
    best = None
    for a in ALPHAS:
        m = Ridge(alpha=a).fit(S.X_tune[S.pos_tune[S.fit_idx]], tune_target[S.fit_idx])
        v = mse(tune_target[S.val_idx], m.predict(S.X_tune[S.pos_tune[S.val_idx]]))
        if best is None or v < best[0]:
            best = (v, a)
    final_model = Ridge(alpha=best[1]).fit(S.X_full[S.pos_full[S.tr_idx]], final_target[S.tr_idx])
    return final_model, best[1]

## 5.5 — Network Architecture, Targeted Fixes & Hyperparameter Tuning

### Mathematical & Implementation Improvements
1. **Isolated Weight Decay:** In `make_optimizer`, `AdamW` applies `weight_decay` **only to the hidden/output weight matrices of the text branch**, setting `weight_decay = 0.0` on `self.skip` and all bias terms. This prevents L2 shrinkage from pulling the persistence coefficient $a$ away from $1.0$.
2. **Full-Window Refit (`train_nn_refit`):**
   - Step 1: Train on `fit_idx` with early stopping on `val_idx` (`patience = 5`) to determine `best_epoch` (and `best_val_mse`).
   - Step 2: Re-initialize with the same seed and train on the **entire training window (`tr_idx`)** for `best_epoch` epochs using `X_full` and `mu_full, sd_full`. This puts the MLP on equal footing with Ridge, especially in the `strict` window.
3. **Robust Loss Option (`loss_type="huber"`):** Replaces squared error with `nn.HuberLoss(delta=HUBER_DELTA)` during training so extreme takeover reports (`logvol_after < -5`, 0.3% of data) do not dominate weight updates. All evaluation remains standard MSE via `scoring.evaluate`.
4. **Within-Year Residual MLP (`mode="within_year"`):** Trains the text MLP directly on within-year demeaned residuals on top of the straight line (`wy_resid`), stripping out macro volatility shifts so the network can only improve by learning **company-level spread**.

In [ ]:
class WordCountNet(nn.Module):
    def __init__(self, n_words, hidden=HIDDEN, dropout=DROPOUT, use_prev=True):
        super().__init__()
        self.use_prev = use_prev
        self.text = nn.Sequential(
            nn.Linear(n_words, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        if use_prev:
            self.skip = nn.Linear(1, 1)
            with torch.no_grad():
                self.skip.weight.fill_(1.0)             # start as "copy last year"
                self.skip.bias.zero_()
                self.text[-1].weight.mul_(0.1)          # word correction starts small
                self.text[-1].bias.zero_()

    def forward(self, x, prev_z=None):
        out = self.text(x).squeeze(1)
        if self.use_prev:
            out = out + self.skip(prev_z.unsqueeze(1)).squeeze(1)
        return out

def make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY):
    """FIX: Apply weight decay only to text-branch weights; exempt skip connection and all biases."""
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("skip.") or name.endswith(".bias"):
            no_decay.append(param)
        else:
            decay.append(param)
    return torch.optim.AdamW([
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=lr)

def make_loss(loss_type="mse"):
    if loss_type == "mse":
        return nn.MSELoss()
    if loss_type == "huber":
        return nn.HuberLoss(delta=HUBER_DELTA)
    raise ValueError(f"Unknown loss_type: {loss_type}")

def dense_from(X, pos, rows):
    return torch.from_numpy(X[pos[rows]].toarray()).to(device)

def predict_raw(model, X, pos, prev_z, rows, batch=1024):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), batch):
            r = rows[i:i + batch]
            pz = torch.from_numpy(prev_z[r]).to(device) if prev_z is not None else None
            out.append(model(dense_from(X, pos, r), pz).cpu().numpy())
    return np.concatenate(out)

def train_nn(S, use_prev, seed, hidden=HIDDEN, dropout=DROPOUT, lr=LR, weight_decay=WEIGHT_DECAY,
             loss_type="mse", train_rows=None, max_epochs=MAX_EPOCHS,
             X=None, pos=None, y_target=None, prev_target=None, scale_sd=None,
             use_val=True, val_X=None, val_pos=None):
    """Core training loop supporting both early-stopping tuning runs and fixed-epoch full-window refits."""
    X = S.X_tune if X is None else X
    pos = S.pos_tune if pos is None else pos
    y_target = S.y_z_fit if y_target is None else y_target
    prev_target = (S.prev_z_fit if use_prev else None) if prev_target is None else prev_target
    scale_sd = S.sd_fit if scale_sd is None else scale_sd
    train_rows = S.fit_idx if train_rows is None else train_rows
    val_X = X if val_X is None else val_X
    val_pos = pos if val_pos is None else val_pos

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = WordCountNet(X.shape[1], hidden=hidden, dropout=dropout, use_prev=use_prev).to(device)
    opt = make_optimizer(model, lr=lr, weight_decay=weight_decay)
    loss_fn = make_loss(loss_type)

    best_val, best_epoch, best_state, wait, history = np.inf, 1, None, 0, []
    for epoch in range(max_epochs):
        t0 = time.perf_counter()
        model.train()
        order_ = rng.permutation(train_rows)
        total = 0.0
        for i in range(0, len(order_), BATCH):
            r = order_[i:i + BATCH]
            opt.zero_grad()
            pz = torch.from_numpy(prev_target[r]).to(device) if use_prev else None
            pred = model(dense_from(X, pos, r), pz)
            loss = loss_fn(pred, torch.from_numpy(y_target[r]).to(device))
            loss.backward()
            opt.step()
            total += loss.item() * len(r)
        train_seconds = time.perf_counter() - t0
        row_log = {"epoch": epoch + 1, "train_loss": total / len(order_), "seconds": train_seconds}

        if use_val:
            val_pred = predict_raw(model, val_X, val_pos, prev_target, S.val_idx)
            val_mse = mse(val_pred, y_target[S.val_idx]) * (scale_sd ** 2)
            row_log["val_mse"] = val_mse
            if val_mse < best_val - 1e-6:
                best_val, best_epoch, wait = val_mse, epoch + 1, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= PATIENCE:
                    history.append(row_log)
                    break
        history.append(row_log)

    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

def train_nn_refit(S, use_prev, seed, hidden=HIDDEN, dropout=DROPOUT, lr=LR,
                   weight_decay=WEIGHT_DECAY, loss_type="mse", refit=True):
    """Tune epoch count on fit_idx -> val_idx, then optionally refit on full tr_idx for best_epoch."""
    model_val, hist_val, best_epoch = train_nn(
        S, use_prev=use_prev, seed=seed, hidden=hidden, dropout=dropout,
        lr=lr, weight_decay=weight_decay, loss_type=loss_type, use_val=True
    )
    if not refit:
        return model_val, hist_val, best_epoch

    model_full, hist_full, _ = train_nn(
        S, use_prev=use_prev, seed=seed, hidden=hidden, dropout=dropout,
        lr=lr, weight_decay=weight_decay, loss_type=loss_type,
        train_rows=S.tr_idx, max_epochs=best_epoch,
        X=S.X_full, pos=S.pos_full,
        y_target=S.y_z_full, prev_target=(S.prev_z_full if use_prev else None),
        scale_sd=S.sd_full, use_val=False
    )
    return model_full, hist_val, best_epoch

def train_within_year_nn(S, seed, hidden=HIDDEN, dropout=DROPOUT, lr=LR,
                         weight_decay=WEIGHT_DECAY, loss_type="mse", refit=True):
    """Train a text-only MLP on within-year demeaned residuals (removing macro year level)."""
    sd_wy_fit = float(S.wy_resid_tune[S.fit_idx].std())
    y_wy_tune_z = (S.wy_resid_tune / sd_wy_fit).astype(np.float32)

    model_val, hist_val, best_epoch = train_nn(
        S, use_prev=False, seed=seed, hidden=hidden, dropout=dropout,
        lr=lr, weight_decay=weight_decay, loss_type=loss_type,
        train_rows=S.fit_idx, X=S.X_tune, pos=S.pos_tune,
        y_target=y_wy_tune_z, prev_target=None, scale_sd=sd_wy_fit, use_val=True
    )
    if not refit:
        return model_val, hist_val, best_epoch, sd_wy_fit

    sd_wy_full = float(S.wy_resid_all[S.tr_idx].std())
    y_wy_full_z = (S.wy_resid_all / sd_wy_full).astype(np.float32)
    model_full, _, _ = train_nn(
        S, use_prev=False, seed=seed, hidden=hidden, dropout=dropout,
        lr=lr, weight_decay=weight_decay, loss_type=loss_type,
        train_rows=S.tr_idx, max_epochs=best_epoch,
        X=S.X_full, pos=S.pos_full,
        y_target=y_wy_full_z, prev_target=None, scale_sd=sd_wy_full, use_val=False
    )
    return model_full, hist_val, best_epoch, sd_wy_full

### Hyperparameter Sweep on Validation Years (Zero Test Leakage)
Before running all 12 splits, we tune `(HIDDEN, DROPOUT, WEIGHT_DECAY)` strictly on the validation years of `MAIN_WINDOW` (using `seed = 0` for speed). The winning configuration is then applied uniformly across all splits and seeds in Section 5.6.

In [ ]:
# Compact validation-only hyperparameter grid for words + last year (paper window validation years)
TUNE_GRID = [
    {"hidden": 64, "dropout": 0.3, "weight_decay": 1e-2},   # original 05-MLP config (with isolated skip decay)
    {"hidden": 32, "dropout": 0.3, "weight_decay": 1e-2},
    {"hidden": 32, "dropout": 0.5, "weight_decay": 5e-2},
    {"hidden": 16, "dropout": 0.5, "weight_decay": 5e-2},
    {"hidden": 16, "dropout": 0.5, "weight_decay": 1e-1},
]

tune_splits = [make_split(*scoring.masks(df, T, MAIN_WINDOW)) for T in scoring.TEST_YEARS]
grid_rows = []
for cfg in TUNE_GRID:
    val_mses, epochs_list = [], []
    for S_t in tune_splits:
        _, hist_t, best_ep = train_nn(
            S_t, use_prev=True, seed=0,
            hidden=cfg["hidden"], dropout=cfg["dropout"], weight_decay=cfg["weight_decay"],
            loss_type="mse", use_val=True
        )
        val_mses.append(hist_t["val_mse"].min())
        epochs_list.append(best_ep)
    grid_rows.append({
        **cfg,
        "mean_val_mse": float(np.mean(val_mses)),
        "mean_best_epoch": float(np.mean(epochs_list)),
    })

tune_df = pd.DataFrame(grid_rows).sort_values("mean_val_mse").reset_index(drop=True)
BEST_CFG = tune_df.iloc[0]
HIDDEN = int(BEST_CFG["hidden"])
DROPOUT = float(BEST_CFG["dropout"])
WEIGHT_DECAY = float(BEST_CFG["weight_decay"])
print(f"Selected config -> HIDDEN={HIDDEN}, DROPOUT={DROPOUT}, WEIGHT_DECAY={WEIGHT_DECAY}")
tune_df.round(5)

## 5.6 — Fitting All Splits & Variants

For every `(window, test_year)` we fit:
- **Baselines:** `train mean`, `copy last year`, `straight line`, `ridge: words only`, `ridge: words + last year` (with expanded `ALPHAS` and leakage-free tuning).
- **MLP Ablations & Variants (3 seeds each, averaged):**
  - `network: words only` (tuned + refitted on full window)
  - `network: words + last year (no refit)` (tuned hyperparameters, held-out `val_year` — isolates the exact effect of refitting)
  - `network: words + last year (refit)` (tuned + refitted on full window)
  - `network: words + last year (Huber, refit)` (robust Huber loss against takeover outliers)
  - `network: within-year words + last year (refit)` (trained on within-year demeaned residuals to prevent macro-year cheating)

In [ ]:
FITS, MODELS, LOG = {}, {}, []

def fit_split(train, test):
    train, test = as_mask(train), as_mask(test)
    key = split_key(train, test)
    if key in FITS:
        return FITS[key]
    t0 = time.perf_counter()
    S = make_split(train, test)
    preds = {"train mean": np.full(len(S.te_idx), y[S.tr_idx].mean())}

    # --- Ridge baselines (expanded ALPHAS, leakage-free validation) ---
    ridge_text, al1 = fit_ridge(S, y, y)
    ridge_resid, al2 = fit_ridge(S, y - S.line_fit, y - S.line_all)
    preds["ridge: words only"] = ridge_text.predict(S.X_full[S.pos_full[S.te_idx]])
    preds["ridge: words + last year"] = S.line_all[S.te_idx] + ridge_resid.predict(S.X_full[S.pos_full[S.te_idx]])

    # --- 1. Words only (refit) ---
    wo_seeds = []
    for seed in SEEDS:
        m, hist, bep = train_nn_refit(S, use_prev=False, seed=seed, hidden=HIDDEN, dropout=DROPOUT, weight_decay=WEIGHT_DECAY, refit=True)
        p = predict_raw(m, S.X_full, S.pos_full, None, S.te_idx) * S.sd_full + S.mu_full
        preds[f"network: words only (refit), seed {seed}"] = p
        wo_seeds.append(p)
        LOG.append({"train_years": f"{S.years[0]}-{S.years[-1]}", "test_year": S.test_year,
                    "model": "network: words only (refit)", "seed": seed,
                    "best_epoch": bep, "best_val_mse": hist["val_mse"].min(), "sec_per_epoch": hist["seconds"].mean()})
    preds[f"network: words only (refit, {len(SEEDS)} seeds averaged)"] = np.mean(wo_seeds, axis=0)

    # --- 2. Words + last year (NO refit — ablation to isolate refit gain) ---
    nr_seeds = []
    for seed in SEEDS:
        m, hist, bep = train_nn_refit(S, use_prev=True, seed=seed, hidden=HIDDEN, dropout=DROPOUT, weight_decay=WEIGHT_DECAY, refit=False)
        # Note: no-refit model was trained on X_tune (cols_tune); transform test with the same fit_mask features
        X_te_nr, pos_te_nr, _ = build_features(S.fit_mask, S.test)
        p = predict_raw(m, X_te_nr, pos_te_nr, S.prev_z_fit, S.te_idx) * S.sd_fit + S.mu_fit
        preds[f"network: words + last year (no refit), seed {seed}"] = p
        nr_seeds.append(p)
    preds[f"network: words + last year (no refit, {len(SEEDS)} seeds averaged)"] = np.mean(nr_seeds, axis=0)

    # --- 3. Words + last year (refit, MSE) ---
    rf_seeds, rf_models = [], []
    for seed in SEEDS:
        m, hist, bep = train_nn_refit(S, use_prev=True, seed=seed, hidden=HIDDEN, dropout=DROPOUT, weight_decay=WEIGHT_DECAY, loss_type="mse", refit=True)
        p = predict_raw(m, S.X_full, S.pos_full, S.prev_z_full, S.te_idx) * S.sd_full + S.mu_full
        preds[f"network: words + last year (refit), seed {seed}"] = p
        rf_seeds.append(p)
        rf_models.append(m)
        LOG.append({"train_years": f"{S.years[0]}-{S.years[-1]}", "test_year": S.test_year,
                    "model": "network: words + last year (refit)", "seed": seed,
                    "best_epoch": bep, "best_val_mse": hist["val_mse"].min(), "sec_per_epoch": hist["seconds"].mean()})
    preds[f"network: words + last year (refit, {len(SEEDS)} seeds averaged)"] = np.mean(rf_seeds, axis=0)

    # --- 4. Words + last year (refit, Huber loss for heavy tails) ---
    hb_seeds = []
    for seed in SEEDS:
        m, hist, bep = train_nn_refit(S, use_prev=True, seed=seed, hidden=HIDDEN, dropout=DROPOUT, weight_decay=WEIGHT_DECAY, loss_type="huber", refit=True)
        p = predict_raw(m, S.X_full, S.pos_full, S.prev_z_full, S.te_idx) * S.sd_full + S.mu_full
        preds[f"network: words + last year (Huber, refit), seed {seed}"] = p
        hb_seeds.append(p)
        LOG.append({"train_years": f"{S.years[0]}-{S.years[-1]}", "test_year": S.test_year,
                    "model": "network: words + last year (Huber, refit)", "seed": seed,
                    "best_epoch": bep, "best_val_mse": hist["val_mse"].min(), "sec_per_epoch": hist["seconds"].mean()})
    preds[f"network: words + last year (Huber, refit, {len(SEEDS)} seeds averaged)"] = np.mean(hb_seeds, axis=0)

    # --- 5. Within-year demeaned residual MLP (refit) ---
    wy_seeds = []
    for seed in SEEDS:
        m, hist, bep, sd_wy = train_within_year_nn(S, seed=seed, hidden=HIDDEN, dropout=DROPOUT, weight_decay=WEIGHT_DECAY, loss_type="mse", refit=True)
        p_resid = predict_raw(m, S.X_full, S.pos_full, None, S.te_idx) * sd_wy
        p = S.line_all[S.te_idx] + p_resid
        preds[f"network: within-year words + last year (refit), seed {seed}"] = p
        wy_seeds.append(p)
        LOG.append({"train_years": f"{S.years[0]}-{S.years[-1]}", "test_year": S.test_year,
                    "model": "network: within-year words + last year (refit)", "seed": seed,
                    "best_epoch": bep, "best_val_mse": hist["val_mse"].min(), "sec_per_epoch": hist["seconds"].mean()})
    preds[f"network: within-year words + last year (refit, {len(SEEDS)} seeds averaged)"] = np.mean(wy_seeds, axis=0)

    MODELS[key] = {
        "nets": rf_models,
        "ridge_text": ridge_text,
        "ridge_resid": ridge_resid,
        "cols": S.cols_full,
        "mu": S.mu_full,
        "sd": S.sd_full,
    }
    FITS[key] = preds
    print(f"train {S.years[0]}-{S.years[-1]} -> test {S.test_year}: {len(S.tr_idx):,} + {len(S.te_idx):,} reports, "
          f"ridge alphas {al1} / {al2}, {time.perf_counter() - t0:.0f}s")
    return preds

In [ ]:
# FIT-ALL: every window and test year
KEY_OF = {}
for window in scoring.WINDOWS:
    for T in scoring.TEST_YEARS:
        train, test = scoring.masks(df, T, window)
        fit_split(train, test)
        KEY_OF[(window, T)] = split_key(train, test)

pd.DataFrame(LOG).groupby(["model", "test_year"]).agg(
    best_epoch=("best_epoch", "mean"),
    sec_per_epoch=("sec_per_epoch", "mean"),
).round(2).T

## 5.7 — Results & Diagnostics

Every model is scored by `scoring.evaluate` against the straight line through last year's volatility (negative `change_pct` is better).

In [ ]:
def lookup(name):
    return lambda df_, train, test: fit_split(train, test)[name]

NET_BASES = [
    "network: words only (refit)",
    "network: words + last year (no refit)",
    "network: words + last year (refit)",
    "network: words + last year (Huber, refit)",
    "network: within-year words + last year (refit)",
]

MODEL_NAMES = ["train mean", "ridge: words only", "ridge: words + last year"]
for base in NET_BASES:
    MODEL_NAMES += [f"{base}, seed {s}" for s in SEEDS]
    MODEL_NAMES += [f"{base[:-1]}, {len(SEEDS)} seeds averaged)"]

results = pd.concat([copy_vs_line] + [scoring.evaluate(df, lookup(n), name=n) for n in MODEL_NAMES], ignore_index=True)
results.to_csv(OUTPUT / "louis-05-wordcount-tuned-results.csv", index=False)

is_seed = results.model.str.contains(r", seed \d+$")
shown = results[~is_seed]
if alex is not None:
    shown = pd.concat([
        shown,
        alex[alex.model.isin(["pooled ridge, Item 7", "within-year ridge, Item 7"])]
            .assign(model=lambda t: t.model + " (FinBERT, from 02)")
    ], ignore_index=True)

summary = (shown.groupby(["model", "window"], sort=False)
           .agg(mean_change_pct=("change_pct", "mean"),
                best_year=("change_pct", "min"),
                worst_year=("change_pct", "max"),
                years_better=("change_pct", lambda c: int((c < 0).sum())),
                years_significant=("ci_high", lambda h: int((h < 0).sum())))
           .round(1))
summary

In [ ]:
# Seed variance across the 3 seeds for each neural variant
seeded = results[is_seed].copy()
seeded["variant"] = seeded.model.str.replace(r", seed \d+$", "", regex=True)
seeded["seed"] = seeded.model.str.extract(r"seed (\d+)$")[0]
(seeded.groupby(["variant", "window", "seed"]).change_pct.mean()
 .groupby(level=["variant", "window"]).agg(mean_change_pct="mean", sd_between_seeds="std").round(2))

In [ ]:
# Figure 1: Year-by-year comparison with 95% paired bootstrap intervals
COLORS = {
    "ridge: words + last year": "#2a78d6",
    f"network: words + last year (refit, {len(SEEDS)} seeds averaged)": "#eb6834",
    f"network: words + last year (Huber, refit, {len(SEEDS)} seeds averaged)": "#8e44ad",
    f"network: within-year words + last year (refit, {len(SEEDS)} seeds averaged)": "#e6a800",
    "within-year ridge, Item 7 (FinBERT, from 02)": "#1baf7a",
}
MARKERS = dict(zip(COLORS, ["o", "s", "^", "v", "D"]))
OFFSET = dict(zip(COLORS, [-0.24, -0.12, 0.0, 0.12, 0.24]))
LABELS = {
    "ridge: words + last year": "ridge on words (tuned)",
    f"network: words + last year (refit, {len(SEEDS)} seeds averaged)": "MLP refit (MSE)",
    f"network: words + last year (Huber, refit, {len(SEEDS)} seeds averaged)": "MLP refit (Huber)",
    f"network: within-year words + last year (refit, {len(SEEDS)} seeds averaged)": "within-year MLP refit",
    "within-year ridge, Item 7 (FinBERT, from 02)": "within-year ridge on FinBERT (02)",
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), sharey=True)
for ax, window in zip(axes, scoring.WINDOWS):
    ax.axhline(0, color="#52514e", lw=1)
    for name in COLORS:
        r = shown[(shown.model == name) & (shown.window == window)]
        if r.empty:
            continue
        pct, lo, hi = r.change_pct, 100 * r.ci_low / r.baseline_mse, 100 * r.ci_high / r.baseline_mse
        ax.errorbar(r.test_year + OFFSET[name], pct, yerr=[pct - lo, hi - pct],
                    fmt=MARKERS[name], color=COLORS[name], ms=5.5, lw=1.4, capsize=0, label=LABELS[name])
    ax.set_title(f"{window} window", fontsize=11)
    ax.set_xticks(list(scoring.TEST_YEARS))
    ax.tick_params(axis="x", labelsize=9)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("change in MSE vs straight line (%)\nbelow 0 = the words help")
axes[-1].legend(frameon=False, loc="best", fontsize=8.5)
fig.suptitle("Figure 1. Tuned & refitted models vs straight line (95% paired bootstrap intervals)",
             fontsize=11, x=0.01, ha="left")
fig.tight_layout()
plt.show()

**Level vs. Spread Decomposition:** Checks whether the tuned/refitted MLPs and the within-year MLP are improving company ranking (*spread*) or merely shifting the annual mean (*level*).

In [ ]:
b_ = copy_vs_line[copy_vs_line.window == MAIN_WINDOW].set_index("test_year")
rows = {"line: level as % of its MSE": 100 * b_["baseline_level"] / b_["baseline_mse"]}
for name in [
    "ridge: words + last year",
    f"network: words + last year (refit, {len(SEEDS)} seeds averaged)",
    f"network: words + last year (Huber, refit, {len(SEEDS)} seeds averaged)",
    f"network: within-year words + last year (refit, {len(SEEDS)} seeds averaged)",
]:
    r = results[(results.model == name) & (results.window == MAIN_WINDOW)].set_index("test_year")
    rows[f"{name}: level"] = 100 * (r["level"] - r["baseline_level"]) / r["baseline_mse"]
    rows[f"{name}: spread"] = 100 * (r["spread"] - r["baseline_spread"]) / r["baseline_mse"]
pd.DataFrame(rows).T.round(1)

**Takeover Sensitivity (Fixed):** Unlike the previous version (which dropped ensemble rows and averaged single-seed MSEs), this table evaluates the **3-seed averaged ensembles** directly so rankings match the main summary table.

In [ ]:
def split_preds(window, T):
    train, test = (as_mask(m) for m in scoring.masks(df, T, window))
    p = dict(fit_split(train, test))
    a, b = scoring.fit_line(df, train)
    p["copy last year"], p["straight line"] = prev[test], a + b * prev[test]
    return y[test], p

rows = []
for window in scoring.WINDOWS:
    for T in scoring.TEST_YEARS:
        yt, p = split_preds(window, T)
        keep = yt >= VERY_LOW
        for name, pt in p.items():
            # FIX: skip individual seed rows and evaluate the 3-seed averaged ensembles directly
            if ", seed " in name:
                continue
            rows.append({
                "window": window,
                "test_year": T,
                "model": name,
                "test_mse": mse(yt, pt),
                "test_mse_excl_takeovers": mse(yt[keep], pt[keep]),
                "takeover_share": 1 - keep.mean(),
            })

tk = (pd.DataFrame(rows).groupby(["window", "model"], sort=False)
      .agg(test_mse=("test_mse", "mean"),
           test_mse_excl_takeovers=("test_mse_excl_takeovers", "mean"),
           reports_dropped=("takeover_share", lambda s: f"{s.mean():.1%}")))
tk.round(3)

In [ ]:
# Predicted vs actual scatter (main window): straight line vs tuned/refitted 3-seed MLP
parts = [split_preds(MAIN_WINDOW, T) for T in scoring.TEST_YEARS]
yt = np.concatenate([p[0] for p in parts])
net_ens_name = f"network: words + last year (refit, {len(SEEDS)} seeds averaged)"

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, name in zip(axes, ["copy last year", net_ens_name]):
    ax.scatter(yt, np.concatenate([p[1][name] for p in parts]), s=4, alpha=0.3)
    lims = [yt.min(), yt.max()]
    ax.plot(lims, lims, color="grey", lw=1)
    ax.axvline(VERY_LOW, color="grey", ls=":", lw=1)
    ax.set_xlabel("actual logvol_after")
    ax.set_title(name, fontsize=10)
axes[0].set_ylabel("predicted logvol_after")
plt.tight_layout()
plt.show()

## 5.8 — Which Words Does It Use? (Ensemble Attribution)

**Fix:** `Gradient × Input` attribution now calls `model.zero_grad()` between batches and averages across **all 3 seeds** of the refitted ensemble rather than inspecting only `seed 0`.

In [ ]:
T_LAST = max(scoring.TEST_YEARS)
M = MODELS[KEY_OF[(MAIN_WINDOW, T_LAST)]]
S_last = make_split(*scoring.masks(df, T_LAST, MAIN_WINDOW))
assert (S_last.cols_full == M["cols"]).all(), "features were rebuilt differently from the fit"
words = vocab[M["cols"]]
word_index = {w: j for j, w in enumerate(words)}
print(f"window {MAIN_WINDOW}, test year {T_LAST}: train {KEY_OF[(MAIN_WINDOW, T_LAST)][0][0]}-{KEY_OF[(MAIN_WINDOW, T_LAST)][0][-1]}")

coef = M["ridge_text"].coef_
order_ = np.argsort(coef)
print("ridge, words only -- push volatility DOWN:", words[order_[:15]].tolist())
print("ridge, words only -- push volatility UP:  ", words[order_[::-1][:15]].tolist())

probe = ["merger", "tender", "acquisition", "bankruptcy", "chapter", "delisted", "default", "going", "concern"]
print("\nwords we expect to matter (ridge coefficient, log-volatility per unit of feature):")
print({w: round(float(coef[word_index[w]]), 3) for w in probe if w in word_index})

In [ ]:
rng = np.random.default_rng(0)
sample = rng.choice(S_last.te_idx, size=min(2000, len(S_last.te_idx)), replace=False)
attr = np.zeros(S_last.X_full.shape[1])

for model in M["nets"]:
    model.eval()
    for i in range(0, len(sample), 500):
        r = sample[i:i + 500]
        model.zero_grad()
        xb = dense_from(S_last.X_full, S_last.pos_full, r).requires_grad_(True)
        pz = torch.from_numpy(S_last.prev_z_full[r]).to(device)
        model(xb, pz).sum().backward()
        attr += (xb.grad * xb).sum(dim=0).detach().cpu().numpy()

attr /= (len(sample) * len(M["nets"]))
order_ = np.argsort(attr)
print("network (3-seed refit ensemble), words + last year -- push prediction DOWN:", words[order_[:15]].tolist())
print("network (3-seed refit ensemble), words + last year -- push prediction UP:  ", words[order_[::-1][:15]].tolist())

## 5.9 — Computational Cost, Parallelism & the 1,000× Scaling Question

To evaluate both **data-volume scaling** and **resource/parallel scaling** (comparing single-threaded vs. multi-threaded execution), we measure:
1. **Learning curve & epoch time** across `10%, 25%, 50%, 100%` of `S_last.fit_idx`.
2. **Strong scaling (CPU threads):** Epoch time on 100% of `S_last.fit_idx` across `1, 2, 4` PyTorch threads.
3. **1,000× back-of-envelope extrapolation** for memory, tokenization, and training epochs.

In [ ]:
# 1. Learning curve and data-volume scaling
rng = np.random.default_rng(0)
rows_ = []
for frac in [0.1, 0.25, 0.5, 1.0]:
    sub = S_last.fit_idx if frac == 1.0 else rng.choice(S_last.fit_idx, size=int(frac * len(S_last.fit_idx)), replace=False)
    t0 = time.perf_counter()
    _, hist, bep = train_nn(S_last, use_prev=True, seed=0, hidden=HIDDEN, dropout=DROPOUT,
                            weight_decay=WEIGHT_DECAY, train_rows=sub, use_val=True)
    rows_.append({
        "fraction": frac, "n_train": len(sub), "best_epoch": bep,
        "val_mse": hist["val_mse"].min(),
        "sec_per_epoch": hist["seconds"].mean(),
        "total_seconds": time.perf_counter() - t0,
    })
lc = pd.DataFrame(rows_)

# 2. Strong scaling: epoch time vs CPU thread count
orig_threads = torch.get_num_threads()
thread_rows = []
for n_th in [1, 2, 4]:
    torch.set_num_threads(n_th)
    _, hist_th, _ = train_nn(S_last, use_prev=True, seed=0, hidden=HIDDEN, dropout=DROPOUT,
                             weight_decay=WEIGHT_DECAY, train_rows=S_last.fit_idx,
                             max_epochs=5, use_val=False)
    thread_rows.append({"threads": n_th, "sec_per_epoch": hist_th["seconds"].mean()})
torch.set_num_threads(orig_threads)
th_df = pd.DataFrame(thread_rows)
th_df["speedup"] = th_df["sec_per_epoch"].iloc[0] / th_df["sec_per_epoch"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(lc["n_train"], lc["val_mse"], "o-")
axes[0].set_xlabel("training reports"); axes[0].set_ylabel("validation MSE"); axes[0].set_title("Learning curve")

axes[1].loglog(lc["n_train"], lc["sec_per_epoch"], "o-")
axes[1].set_xlabel("training reports"); axes[1].set_ylabel("seconds per epoch"); axes[1].set_title("Data-volume scaling")

axes[2].plot(th_df["threads"], th_df["speedup"], "s-", color="#2a78d6")
axes[2].plot(th_df["threads"], th_df["threads"], "--", color="grey", lw=1, label="ideal linear")
axes[2].set_xlabel("CPU threads"); axes[2].set_ylabel("speedup vs 1 thread"); axes[2].set_title("Strong scaling (CPU)")
axes[2].legend(frameon=False)
plt.tight_layout()
plt.show()

slope_time = np.polyfit(np.log(lc["n_train"]), np.log(lc["sec_per_epoch"]), 1)[0]
print(f"log-log slope of seconds per epoch vs training reports: {slope_time:.2f} (1.0 = linear)")
display(lc.round(4))
display(th_df.round(3))

In [ ]:
# 1,000x back-of-envelope extrapolation
FACTOR = 1000
full = lc.iloc[-1]
csr_bytes = S_last.X_full.data.nbytes + S_last.X_full.indices.nbytes + S_last.X_full.indptr.nbytes
n_params = sum(p.numel() for p in M["nets"][0].parameters())
count_secs = timings["count_seconds"]

print(f"reports: {len(df):,}  ->  {len(df) * FACTOR:,.0f}")
print(f"sparse feature table (one split): {csr_bytes / 1e6:,.0f} MB  ->  {csr_bytes * FACTOR / 1e9:,.0f} GB "
      f"({csr_bytes / S_last.X_full.shape[0] / 1e3:.1f} kB per report)")
print(f"counting words (one pass, one core): {count_secs / 60:,.1f} min  ->  {count_secs * FACTOR / 3600:,.0f} hours")
print(f"one training epoch: {full['sec_per_epoch']:.3f} s  ->  {full['sec_per_epoch'] * FACTOR / 3600:,.2f} hours (linear)")
print(f"network parameters: {n_params:,} (fixed by N_WORDS={N_WORDS} and HIDDEN={HIDDEN}, independent of N)")

## 5.10 — Conclusions & Interpretations *(To Be Written After Running)*

- **Effect of refitting & tuning:** Compare `no refit` vs. `refit` across `paper` and `strict` windows, and note whether the expanded `ALPHAS` changed the Ridge benchmark.
- **Level vs. spread & within-year MLP:** Does training on within-year demeaned residuals reduce downside risk and improve company-level spread relative to `within-year ridge (FinBERT)`?
- **Heavy-tail robustness (Huber loss):** Did `HuberLoss` reduce test MSE or seed variance, and how did it behave with vs. without takeover reports (`logvol_after < -5`)?
- **Scaling takeaways:** Summarize the data-volume slope, CPU thread speedup, and why I/O + tokenization dominate at 1,000× scale (motivating Spark/Dask map-reduce or stateless `HashingVectorizer` + multi-GPU data parallelism).